# ⚡ ClipForge AI — Free GPU Worker (Google Colab)

Welcome to the **ClipForge AI Worker Notebook**.
This notebook connects to your ClipForge queue, claims queued video clipping jobs, performs GPU-accelerated transcription (Faster-Whisper FP16), heuristic viral highlight analysis, and synthesizes 1080x1920 9:16 vertical short clips with blurred background padding and styled titles.

---

### 🛡️ Resilience & ₹0 Budget Architecture
- **Resumable Chunks & Checkpointing**: Videos are processed in 120-second chunks. Each chunk is saved directly to your Google Drive and registered in the database.
- **Safe Disconnection Recovery**: If Colab disconnects or times out, your generated clips and progress are **never lost**. Simply re-run this notebook, and the worker will pick up right where it left off, skipping already finished chunks!
- **Zero Fake Keep-Alive**: Complies strictly with Google Colab Terms of Service. No artificial activity or auto-clickers.

## 📦 Step 1: Single-Cell Setup (Mount Google Drive & Install Tools)
Click the **Play (Run)** button below to mount Google Drive and install `faster-whisper`, `yt-dlp`, and `ffmpeg`.

In [ ]:
# @title Setup Environment & Mount Storage
import os
import sys
import torch
from google.colab import drive

# 1. Mount Google Drive for persistent checkpointing & clips storage
print("📁 Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Create ClipForge storage hierarchy
BASE_STORAGE = "/content/drive/MyDrive/ClipForge"
for sub in ["videos", "clips", "checkpoints", "models", "logs", "exports"]:
    os.makedirs(os.path.join(BASE_STORAGE, sub), exist_ok=True)
print(f"✅ Persistent storage initialized at: {BASE_STORAGE}")

# 3. Install GPU-accelerated dependencies
print("🚀 Installing faster-whisper, yt-dlp, and ffmpeg-python...")
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q faster-whisper yt-dlp requests imageio-ffmpeg pydantic

# 4. Check GPU Availability
device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if device == "cuda" else "CPU only"
print(f"🎮 Compute Hardware Detected: {gpu_name} (Mode: {device.upper()})")

## 🚀 Step 2: Start ClipForge Worker
Enter your Backend API URL (or Ngrok / Cloudflare Tunnel URL) below and run this cell.
The worker will continuously process waiting jobs and idle safely when the queue is empty.

In [ ]:
# @title Start Processing Jobs
# Enter your deployed backend URL or tunnel endpoint (e.g., https://your-domain.com or https://xxxx.trycloudflare.com)
API_URL = "http://your-backend-api-url.com" # @param {type:"string"}
GITHUB_REPO_URL = "https://github.com/Somil-Jain24/clipforge.git" # @param {type:"string"}
WORKER_NAME = "colab-gpu-worker-1" # @param {type:"string"}
WHISPER_MODEL = "base" # @param ["tiny", "base", "small", "medium"]

import os
import sys
import time
import requests

# Clone or update repo from GitHub, or use existing local copy
if os.path.exists("/content/worker"):
    sys.path.append("/content")
elif os.path.exists("/content/drive/MyDrive/ClipForge/worker"):
    sys.path.append("/content/drive/MyDrive/ClipForge")
else:
    !git clone https://github.com/Somil-Jain24/clipforge.git /content/clipforge_repo 2>/dev/null || (cd /content/clipforge_repo && git pull)
    sys.path.append("/content/clipforge_repo")

from worker.runner import ClipForgeWorker

print(f"✨ Connecting to ClipForge Queue at: {API_URL}")
worker = ClipForgeWorker(
    api_url=API_URL,
    worker_id=WORKER_NAME,
    chunk_duration_sec=120,
    whisper_model=WHISPER_MODEL
)

# Run continuous processing loop
try:
    worker.start(single_job_mode=False)
except KeyboardInterrupt:
    print("\n🛑 Worker stopped by user. Any completed checkpoints remain safely stored on Google Drive.")